# Handwriting neural separability analysis
## Willett et al. 2021 (Nature) — Extended Data Figure 1 reproduction

**Paper**: *High-performance brain-to-text communication via handwriting*

**Pipeline**:
1. Data loading and preprocessing (per-block z-score + Gaussian smoothing)
2. Dataset structure exploration
3. PCA neural trajectories (raw vs time-aligned)
4. Pen-tip velocity ↔ neural activity correspondence
5. Per-trial t-SNE (all 10 sessions)
6. k-NN classification accuracy (LOO, all sessions)

**Key parameters (validated on data)**:
- Go cue at cube bin 51 (= 510 ms marker)
- Movement window: bins [51, 151] → 0–1000 ms after go cue
- Features: 100 bins × 192 channels flattened → PCA(100) → k-NN

## 0. Imports & path configuration

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.gridspec import GridSpec
import scipy.io as sio
from scipy.ndimage import gaussian_filter1d
from scipy.interpolate import interp1d
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import LeaveOneOut
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

%matplotlib inline
plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.size"]  = 10
print("Imports loaded.")

In [ ]:
# Path configuration
BASE         = "./"
DATA_DIR     = os.path.join(BASE, "doi_10_5061_dryad_wh70rxwmv__v20211015", "handwritingBCIData")
DATASETS_DIR = os.path.join(DATA_DIR, "Datasets")
WARP_DIR     = os.path.join(DATA_DIR, "RNNTrainingSteps", "Step1_TimeWarping")
TMPL_PATH    = os.path.join(DATASETS_DIR, "computerMouseTemplates.mat")
OUT_DIR      = os.path.join(BASE, "figures")
os.makedirs(OUT_DIR, exist_ok=True)

PRIMARY_SESSION = "t5.2019.05.08"
ALL_SESSIONS    = sorted([
    d for d in os.listdir(DATASETS_DIR)
    if d.startswith("t5.") and
       os.path.exists(os.path.join(DATASETS_DIR, d, "singleLetters.mat"))
])
LETTERS = list("abcdefghijklmnopqrstuvwxyz")

# Verified: go cue at cube bin 51 (sliding-window matches raw timeline)
GO_BIN   = 51
MOVE_WIN = (GO_BIN, GO_BIN + 100)   # 0–1000 ms after go cue

print(f"Found {len(ALL_SESSIONS)} session(s):")
for s in ALL_SESSIONS:
    print(f"  {s}")

## 1. Dataset structure

Inspect fields in `singleLetters.mat`, and the shape / value range of neural activity cubes.

In [ ]:
# Load raw .mat and inspect structure
sl_raw = sio.loadmat(
    os.path.join(DATASETS_DIR, PRIMARY_SESSION, "singleLetters.mat"),
    simplify_cells=True
)

non_meta = [k for k in sl_raw if not k.startswith("_")]
cube_keys = [k for k in non_meta if k.startswith("neuralActivityCube_")]
other_keys = [k for k in non_meta if not k.startswith("neuralActivityCube_")]

print(f"=== {PRIMARY_SESSION}/singleLetters.mat ===\n")
print(f"Neural activity cubes ({len(cube_keys)} letters):")
ex = sl_raw["neuralActivityCube_a"]
print(f"  shape: {ex.shape}  ->  (trials, time_bins, channels)")
print(f"  dtype: {ex.dtype},  range: [{ex.min()}, {ex.max()}]  (raw threshold crossings, not preprocessed)\n")

print("Other fields:")
for k in other_keys:
    v = sl_raw[k]
    if hasattr(v, "shape"):
        print(f"  {k:<30s} shape={v.shape}  dtype={v.dtype}")
    else:
        print(f"  {k:<30s} {type(v).__name__}: {str(v)[:80]}")

In [ ]:
# Plot: trial-averaged firing rate vs time for all letters (raw, not preprocessed)
# Shows go cue location in the cube

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle("Raw neural features (not preprocessed)", fontsize=12)

# Left: trial-averaged rate on most active channel
most_active_ch = np.array([
    sl_raw[f"neuralActivityCube_{l}"].mean(axis=(0, 1))
    for l in LETTERS if f"neuralActivityCube_{l}" in sl_raw
]).mean(axis=0).argmax()

t_bins = np.arange(201) * 10   # ms
colors_l = cm.tab20(np.linspace(0, 1, 26))
for i, l in enumerate(LETTERS):
    key = f"neuralActivityCube_{l}"
    if key not in sl_raw: continue
    avg = sl_raw[key][:, :, most_active_ch].mean(axis=0)   # (201,)
    axes[0].plot(t_bins, avg, color=colors_l[i], alpha=0.6, linewidth=0.9, label=l)
axes[0].axvline(GO_BIN * 10, color="k", linestyle="--", linewidth=1.5, label=f"go cue (bin {GO_BIN})")
axes[0].set(xlabel="Time (ms)", ylabel="Spike count / 10 ms bin",
            title=f"All letters, trial mean (ch {most_active_ch}, most active)")
axes[0].legend(fontsize=6, ncol=4, loc="upper right")

# Right: mean firing rate per channel (Hz)
ch_rates_hz = sl_raw["neuralActivityTimeSeries"].astype(float).mean(axis=0) * 100
axes[1].hist(ch_rates_hz, bins=30, color="steelblue", edgecolor="white")
axes[1].axvline(2, color="red", linestyle="--", label="2 Hz activity threshold")
axes[1].set(xlabel="Mean firing rate (Hz)", ylabel="# channels",
            title="192-channel firing rate distribution")
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "explore_raw_data.png"), dpi=150)
plt.show()

print(f"Most active channel: {most_active_ch}")
print(f"Channels with mean rate >= 2 Hz: {(ch_rates_hz >= 2).sum()} / {len(ch_rates_hz)}")
print(f"Mean firing rate across channels: {ch_rates_hz.mean():.1f} Hz")

## 2. Preprocessing

**Three steps** (as in the paper):
1. **Subtract per-trial block mean** — use `meansPerBlock` for each trial's recording block to remove slow drift
2. **Divide by global std** — `stdAcrossAllData` to normalize channels
3. **Gaussian smoothing** — sigma = 20 ms (= 2 bins) to reduce Poisson noise in spike counts

Data are stored as raw uint8 spike counts; **not** preprocessed inside the dataset files.

In [ ]:
def load_and_preprocess(session, sigma_ms=20.0, dt_ms=10.0):
    """
    Returns:
      cubes : dict  letter -> (trials, 201, 192) float32, z-scored + smoothed
      sl    : raw mat-file dict (kept for plotting)
    """
    sl   = sio.loadmat(os.path.join(DATASETS_DIR, session, "singleLetters.mat"),
                       simplify_cells=True)
    means      = sl["meansPerBlock"]                         # (n_blocks, 192)
    global_std = sl["stdAcrossAllData"].astype(np.float32)   # (192,)
    global_std = np.where(global_std == 0, 1.0, global_std)
    block_nums = sl["blockNumsTimeSeries"]                   # (T_total,)
    block_list = sl["blockList"]                             # (n_blocks,)
    go_bins    = sl["goPeriodOnsetTimeBin"]                  # (n_trials_all,)
    char_cues  = sl["characterCues"]                         # (n_trials_all,)
    bml        = {b: means[i] for i, b in enumerate(block_list)}
    sigma_bins = sigma_ms / dt_ms

    cubes = {}
    for letter in LETTERS:
        key = f"neuralActivityCube_{letter}"
        if key not in sl:
            continue
        trial_idx = np.where(char_cues == letter)[0]
        # per-trial block mean -> (trials, 192)
        tbm = np.array([
            bml.get(block_nums[min(go_bins[i], len(block_nums)-1)], means.mean(0))
            for i in trial_idx
        ], dtype=np.float32)
        cube = sl[key].astype(np.float32)                    # (trials, 201, 192)
        cube = (cube - tbm[:, None, :]) / global_std[None, None, :]
        cube = gaussian_filter1d(cube, sigma=sigma_bins, axis=1)
        cubes[letter] = cube
    return cubes, sl

def trial_avg(cubes):
    return {l: cubes[l].mean(axis=0) for l in cubes}

def movement_features(cube, flatten=True):
    """Movement window features: bins [GO_BIN, GO_BIN+100]"""
    w = cube[:, MOVE_WIN[0]:MOVE_WIN[1], :]   # (trials, 100, 192)
    return w.reshape(w.shape[0], -1) if flatten else w.mean(axis=1)

print("Preprocessing helpers defined.")

In [ ]:
# Load primary session and preprocess
print(f"Loading and preprocessing: {PRIMARY_SESSION} ...")
cubes_raw, sl_primary = load_and_preprocess(PRIMARY_SESSION)
n_trials, n_time, n_chan = cubes_raw["a"].shape
avg_raw = trial_avg(cubes_raw)

print(f"\nPreprocessed cube shape: {n_trials} trials x {n_time} bins x {n_chan} channels")
print(f"Example value range (letter 'a'): min={cubes_raw['a'].min():.2f}, "
      f"max={cubes_raw['a'].max():.2f}, mean={cubes_raw['a'].mean():.4f}")

# Before vs after: one letter, one channel
ch_show = most_active_ch
letter_show = "a"
raw_trial0 = sl_primary[f"neuralActivityCube_{letter_show}"][0, :, ch_show].astype(float)
proc_trial0 = cubes_raw[letter_show][0, :, ch_show]

fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
fig.suptitle(f"Preprocessing comparison — letter '{letter_show}', channel {ch_show}", fontsize=11)
t_ms = np.arange(n_time) * 10
axes[0].plot(t_ms, raw_trial0, color="steelblue", linewidth=1.2, label="raw spike count")
axes[0].axvline(GO_BIN*10, color="red", linestyle="--", alpha=0.7, label="go cue")
axes[0].set(xlabel="Time (ms)", ylabel="Spike count / 10 ms", title="Raw (unprocessed)")
axes[0].legend()

axes[1].plot(t_ms, proc_trial0, color="darkorange", linewidth=1.2, label="z-score + smooth")
axes[1].axvline(GO_BIN*10, color="red", linestyle="--", alpha=0.7, label="go cue")
axes[1].axvspan(MOVE_WIN[0]*10, MOVE_WIN[1]*10, alpha=0.12, color="green", label="movement window")
axes[1].set(xlabel="Time (ms)", ylabel="Z-score", title="After preprocessing (z-score + 20 ms smooth)")
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "preprocess_comparison.png"), dpi=150)
plt.show()

In [ ]:
# Trial-to-trial variability: letter 'a' vs trial mean
fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
fig.suptitle("Trial-to-trial variability — letter 'a'", fontsize=11)
t_ms = np.arange(n_time) * 10

# Left: single-trial traces (most active channel)
for i in range(n_trials):
    axes[0].plot(t_ms, cubes_raw["a"][i, :, ch_show],
                 color="steelblue", alpha=0.25, linewidth=0.7)
axes[0].plot(t_ms, avg_raw["a"][:, ch_show],
             color="navy", linewidth=2, label="trial mean")
axes[0].axvline(GO_BIN*10, color="red", linestyle="--", alpha=0.7, label="go cue")
axes[0].axvspan(MOVE_WIN[0]*10, MOVE_WIN[1]*10, alpha=0.1, color="green")
axes[0].set(xlabel="Time (ms)", ylabel="Z-score",
            title=f"Channel {ch_show}: {n_trials} trials (faint) + mean")
axes[0].legend()

# Right: trial-mean heatmap (all channels, movement window)
mov_avg = avg_raw["a"][MOVE_WIN[0]:MOVE_WIN[1], :]   # (100, 192)
im = axes[1].imshow(mov_avg.T, aspect="auto", origin="lower",
                    extent=[0, 1000, 0, n_chan], cmap="RdBu_r")
axes[1].set(xlabel="Time after go cue (ms)", ylabel="Channel",
            title="Trial-mean heatmap (movement window, all channels)")
plt.colorbar(im, ax=axes[1], label="Z-score")

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "trial_variability.png"), dpi=150)
plt.show()

## 3. PCA neural trajectories

PCA on trial-averaged activity for all letters; plot time trajectories in the first 3 PCs.

- **Left**: raw (not time-aligned) — cross-trial timing variability blurs trajectories
- **Right**: time-aligned (DTW warped) — more consistent, separable trajectories per letter

In [ ]:
# Load time-aligned warpedCubes and trial means
print("Loading warpedCubes ...")
warp_mat = sio.loadmat(
    os.path.join(WARP_DIR, f"{PRIMARY_SESSION}_warpedCubes.mat"),
    simplify_cells=True
)
session_mean  = sl_primary["meansPerBlock"].mean(axis=0)
global_std_p  = np.where(sl_primary["stdAcrossAllData"] == 0, 1.0,
                         sl_primary["stdAcrossAllData"])

avg_warp = {}
for l in LETTERS:
    if l not in warp_mat: continue
    w   = warp_mat[l].astype(np.float64)           # (trials, 201, 192), may have NaN
    avg = np.nanmean(w, axis=0)                     # (201, 192)
    avg = (avg - session_mean[None, :]) / global_std_p[None, :]
    avg = gaussian_filter1d(avg, sigma=2.0, axis=0)
    avg_warp[l] = avg

print(f"warpedCubes loaded; NaN fraction example (letter 'a'):")
nan_frac = np.isnan(warp_mat["a"]).mean()
print(f"  overall NaN fraction: {nan_frac*100:.1f}% (where DTW stretch goes out of bounds)")

In [ ]:
def fit_pca_trajectories(avg_dict, letters, n_components=3):
    X   = np.vstack([avg_dict[l] for l in letters if l in avg_dict])
    pca = PCA(n_components=n_components).fit(X)
    return pca, {l: pca.transform(avg_dict[l]) for l in letters if l in avg_dict}

pca_raw,  proj_raw  = fit_pca_trajectories(avg_raw,  LETTERS)
pca_warp, proj_warp = fit_pca_trajectories(avg_warp, LETTERS)

print("Explained variance (raw):   " + "  ".join(
    f"PC{i+1}={v*100:.1f}%" for i, v in enumerate(pca_raw.explained_variance_ratio_)))
print("Explained variance (warped):" + "  ".join(
    f"PC{i+1}={v*100:.1f}%" for i, v in enumerate(pca_warp.explained_variance_ratio_)))

In [ ]:
SHOW = ["a", "e", "i", "o", "u", "t"]
colors_show = cm.tab10(np.linspace(0, 1, len(SHOW)))

fig = plt.figure(figsize=(14, 6))
fig.suptitle("PCA neural trajectories — trial-mean activity (first 3 PCs)", fontsize=13)

for col, (proj, title, pca_obj) in enumerate([
        (proj_raw,  "Raw (not time-aligned)", pca_raw),
        (proj_warp, "Time-aligned (DTW warped)", pca_warp)]):
    ax = fig.add_subplot(1, 2, col + 1, projection="3d")
    for i, l in enumerate(SHOW):
        traj = proj[l]
        ax.plot(traj[:, 0], traj[:, 1], traj[:, 2],
                color=colors_show[i], label=l, linewidth=2, alpha=0.85)
        # Trajectory start (delay onset)
        ax.scatter(*traj[0], color=colors_show[i], s=40, zorder=5)
        # Go cue
        ax.scatter(*traj[GO_BIN], color=colors_show[i], s=60,
                   marker="*", zorder=6, edgecolors="k", linewidths=0.5)
    v = pca_obj.explained_variance_ratio_
    ax.set_xlabel(f"PC1 ({v[0]*100:.1f}%)", labelpad=6)
    ax.set_ylabel(f"PC2 ({v[1]*100:.1f}%)", labelpad=6)
    ax.set_zlabel(f"PC3 ({v[2]*100:.1f}%)", labelpad=6)
    ax.set_title(title)
    ax.legend(fontsize=8, loc="upper left")
    ax.text2D(0.02, 0.02, "dot=start  star=go cue", transform=ax.transAxes, fontsize=7)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "fig1_pca_trajectories.png"), dpi=150)
plt.show()

In [ ]:
# All 26 letters, PCA trajectories (PC1 vs PC2)
colors_all26 = cm.tab20(np.linspace(0, 1, 26))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("All 26 letters: PCA trajectories (PC1 vs PC2)", fontsize=12)

for ax, proj, title in [(axes[0], proj_raw, "Raw"), (axes[1], proj_warp, "Time-aligned")]:
    for i, l in enumerate(LETTERS):
        if l not in proj: continue
        traj = proj[l]
        ax.plot(traj[:, 0], traj[:, 1],
                color=colors_all26[i], linewidth=1.2, alpha=0.75, label=l)
        ax.scatter(traj[GO_BIN, 0], traj[GO_BIN, 1],
                   color=colors_all26[i], s=25, zorder=4)
    ax.set(xlabel="PC1", ylabel="PC2", title=title)
    ax.legend(fontsize=6, ncol=4, loc="lower right")

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "fig1b_pca_2d_all_letters.png"), dpi=150)
plt.show()

## 4. Pen velocity ↔ neural activity

Ridge regression from trial-averaged neural activity to pen-velocity templates; report R².

Paper: velocity explains ~**30%** of total neural variance — motor cortex encodes handwriting.

In [ ]:
tmpl = sio.loadmat(TMPL_PATH, simplify_cells=True)
print("Letters in computerMouseTemplates.mat:",
      [k for k in tmpl if not k.startswith("_") and k != "dataDescription"])
print("\nVelocity template description:", tmpl.get("dataDescription", ""))

def resample_template(vel, n_out):
    f = interp1d(np.linspace(0, 1, vel.shape[0]), vel, axis=0)
    return f(np.linspace(0, 1, n_out))

# Example velocity templates (variable length before resampling)
fig, axes = plt.subplots(2, 3, figsize=(13, 5))
fig.suptitle("Pen-tip velocity templates (mouse drawing, 100 Hz)", fontsize=11)
for ax, l in zip(axes.flat, ["a", "b", "e", "m", "t", "z"]):
    vel = tmpl[l]   # (T_orig, 2)
    t   = np.arange(vel.shape[0]) * 10   # ms at 100Hz
    ax.plot(t, vel[:, 0], "b-",  linewidth=1.4, label="vx")
    ax.plot(t, vel[:, 1], "r-",  linewidth=1.4, label="vy")
    ax.set(title=f"Letter '{l}'", xlabel="ms", ylabel="velocity (a.u.)")
    ax.legend(fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "velocity_templates.png"), dpi=150)
plt.show()

In [ ]:
# Ridge: neural activity -> pen velocity; per-letter R²
r2_per_letter = {}
vel_decoded   = {}
t_ax = np.arange(n_time) * 10   # ms

for letter in LETTERS:
    if letter not in avg_raw or letter not in tmpl: continue
    vel    = resample_template(tmpl[letter], n_time)   # (201, 2)
    neural = avg_raw[letter]                            # (201, 192)
    reg    = Ridge(alpha=1.0).fit(neural, vel)
    vel_p  = reg.predict(neural)
    r2_per_letter[letter] = r2_score(vel, vel_p, multioutput="variance_weighted")
    vel_decoded[letter]   = (vel, vel_p)

# Global R²: stack all letters
all_neural = np.vstack([avg_raw[l] for l in LETTERS if l in avg_raw])
all_vel    = np.vstack([resample_template(tmpl[l], n_time)
                        for l in LETTERS if l in avg_raw and l in tmpl])
total_r2   = r2_score(all_vel,
                      Ridge(alpha=1.0).fit(all_neural, all_vel).predict(all_neural),
                      multioutput="variance_weighted")

mean_r2 = np.mean(list(r2_per_letter.values()))
print(f"Mean per-letter R²: {mean_r2:.3f}")
print(f"Fraction of total neural variance explained by pen velocity: {total_r2*100:.1f}%  (paper ~30%)")

In [ ]:
letters_s = sorted(r2_per_letter)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Pen velocity ↔ neural activity", fontsize=12)

# Left: R² bar chart
axes[0].bar(letters_s, [r2_per_letter[l] for l in letters_s],
            color="steelblue", alpha=0.8)
axes[0].axhline(mean_r2, color="red", linestyle="--",
                label=f"mean R²={mean_r2:.2f}")
axes[0].set(xlabel="Letter", ylabel="R²", title="Per-letter R² (neural -> pen velocity)", ylim=(0, 1))
axes[0].legend()

# Center: decoded velocity for 'a'
vel_t, vel_p = vel_decoded["a"]
for vt, vp, col, lbl in zip(vel_t.T, vel_p.T, ["b", "r"], ["vx", "vy"]):
    axes[1].plot(t_ax, vt, f"{col}-",  linewidth=1.4, label=f"true {lbl}")
    axes[1].plot(t_ax, vp, f"{col}--", linewidth=1.4, alpha=0.8, label=f"decoded {lbl}")
axes[1].axvline(GO_BIN*10, color="k", linestyle=":", alpha=0.6)
axes[1].axvspan(MOVE_WIN[0]*10, MOVE_WIN[1]*10, alpha=0.08, color="green")
axes[1].set(xlabel="Time (ms)", ylabel="velocity (a.u.)", title="Decoded pen velocity — letter 'a'")
axes[1].legend(fontsize=8)

# Right: decoded velocity for 't'
vel_t, vel_p = vel_decoded["t"]
for vt, vp, col, lbl in zip(vel_t.T, vel_p.T, ["b", "r"], ["vx", "vy"]):
    axes[2].plot(t_ax, vt, f"{col}-",  linewidth=1.4, label=f"true {lbl}")
    axes[2].plot(t_ax, vp, f"{col}--", linewidth=1.4, alpha=0.8, label=f"decoded {lbl}")
axes[2].axvline(GO_BIN*10, color="k", linestyle=":", alpha=0.6)
axes[2].axvspan(MOVE_WIN[0]*10, MOVE_WIN[1]*10, alpha=0.08, color="green")
axes[2].set(xlabel="Time (ms)", ylabel="velocity (a.u.)", title="Decoded pen velocity — letter 't'")
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "fig2_pen_velocity.png"), dpi=150)
plt.show()

## 5. Load all 10 sessions

t-SNE and k-NN need more data; load every session and extract movement-window features.

**Features**: each trial -> movement window (100 bins x 192 channels) flattened = 19,200-D vector

In [ ]:
print("Loading all sessions ...")
X_all, y_all = [], []
session_trial_counts = {}

for sess in ALL_SESSIONS:
    cubes_s, _ = load_and_preprocess(sess)
    n_sess = 0
    for letter in LETTERS:
        if letter not in cubes_s: continue
        feats = movement_features(cubes_s[letter], flatten=True)   # (trials, 19200)
        X_all.append(feats)
        y_all.extend([letter] * feats.shape[0])
        n_sess += feats.shape[0]
    session_trial_counts[sess] = n_sess
    print(f"  {sess}: {n_sess} trials")

X_pool = np.vstack(X_all)      # (3042, 19200)
y_pool = np.array(y_all)

print(f"\nPooled data: {len(y_pool)} trials x {X_pool.shape[1]} features")
print(f"Trials per letter: {dict(zip(*np.unique(y_pool, return_counts=True)))}")

## 6. t-SNE visualization

PCA to 30-D (denoise + speed), then t-SNE to 2-D.

If handwriting representations are separable, trials of the same letter should form tight clusters.

In [ ]:
print("PCA (19200 -> 30) ...")
X_pca30 = PCA(n_components=30, random_state=42).fit_transform(X_pool)
print(f"PCA done, shape: {X_pca30.shape}")

print("Running t-SNE (may take ~1 min) ...")
# sklearn>=1.5: TSNE uses max_iter; older versions use n_iter (otherwise this cell fails and Z is never set)
_tsne_kw = dict(n_components=2, perplexity=30, random_state=42, learning_rate="auto", init="pca")
try:
    tsne = TSNE(**_tsne_kw, max_iter=1000)
except TypeError:
    tsne = TSNE(**_tsne_kw, n_iter=1000)
Z = tsne.fit_transform(X_pca30)
print(f"t-SNE done, KL divergence: {tsne.kl_divergence_:.4f}")

In [ ]:
colors_26  = cm.tab20(np.linspace(0, 1, 26))
letter_to_i = {l: i for i, l in enumerate(LETTERS)}

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle("t-SNE — single-trial neural activity (10 sessions, 3042 trials)", fontsize=12)

# Left: all letters
ax = axes[0]
for l in LETTERS:
    mask = y_pool == l
    ax.scatter(Z[mask, 0], Z[mask, 1],
               color=colors_26[letter_to_i[l]],
               label=l, s=12, alpha=0.65)
ax.set(title="All 26 letters", xlabel="t-SNE dim 1", ylabel="t-SNE dim 2")
ax.legend(fontsize=6.5, ncol=5, loc="best", markerscale=1.8)

# Right: vowels only
ax2 = axes[1]
vowels = list("aeiou")
for i, l in enumerate(vowels):
    mask = y_pool == l
    ax2.scatter(Z[mask, 0], Z[mask, 1],
                color=cm.tab10(i / 10),
                label=l, s=25, alpha=0.8)
ax2.set(title="Vowels only (a/e/i/o/u)", xlabel="t-SNE dim 1", ylabel="t-SNE dim 2")
ax2.legend(fontsize=9, markerscale=1.8)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "fig3_tsne.png"), dpi=150)
plt.show()

## 7. k-NN classification (leave-one-out)

**Setup** (paper):
- Features: movement window flattened (19,200-D) -> PCA(100)
- k=1, Euclidean distance
- LOO across all trials (training always includes all other sessions)

Paper accuracy: **94.1%** (on DTW time-aligned features in the original work)

In [ ]:
print("PCA (19200 -> 100) ...")
X_pca100 = PCA(n_components=100, random_state=42).fit_transform(X_pool)

print(f"Running k-NN LOO ({len(y_pool)} trials; may take several minutes)...")
knn = KNeighborsClassifier(n_neighbors=1, metric="euclidean")
correct = 0
for train_i, test_i in LeaveOneOut().split(X_pca100):
    knn.fit(X_pca100[train_i], y_pool[train_i])
    correct += int(knn.predict(X_pca100[test_i])[0] == y_pool[test_i[0]])

accuracy = correct / len(y_pool)
print(f"\nOK k-NN (k=1) LOO accuracy: {accuracy*100:.1f}%  (paper: 94.1%)")

In [ ]:
# Per-letter LOO accuracy
per_letter_acc = {}
for letter in LETTERS:
    mask       = y_pool == letter
    letter_idx = np.where(mask)[0]
    other_idx  = np.where(~mask)[0]
    correct_l  = 0
    for ti in letter_idx:
        train_idx = np.concatenate([other_idx, letter_idx[letter_idx != ti]])
        knn.fit(X_pca100[train_idx], y_pool[train_idx])
        correct_l += int(knn.predict(X_pca100[[ti]])[0] == letter)
    per_letter_acc[letter] = correct_l / len(letter_idx)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle(f"k-NN accuracy (k=1 LOO, all sessions) — overall {accuracy*100:.1f}%", fontsize=12)

# Bar chart
bars = axes[0].bar(list(per_letter_acc), [per_letter_acc[l] for l in per_letter_acc],
                   color=[cm.RdYlGn(per_letter_acc[l]) for l in per_letter_acc])
axes[0].axhline(accuracy, color="red", linestyle="--", linewidth=1.5,
                label=f"overall LOO = {accuracy*100:.1f}%")
axes[0].axhline(1/26, color="gray", linestyle=":", linewidth=1,
                label=f"chance = {100/26:.1f}%")
axes[0].set(xlabel="Letter", ylabel="accuracy",
            title="Per-letter accuracy", ylim=(0, 1.05))
axes[0].legend(fontsize=8)

# Sorted
sorted_letters = sorted(per_letter_acc, key=per_letter_acc.get)
sorted_accs    = [per_letter_acc[l] for l in sorted_letters]
colors_grad    = [cm.RdYlGn(a) for a in sorted_accs]
axes[1].barh(sorted_letters, sorted_accs, color=colors_grad)
axes[1].axvline(accuracy, color="red", linestyle="--", linewidth=1.5)
axes[1].axvline(1/26, color="gray", linestyle=":", linewidth=1)
axes[1].set(xlabel="accuracy", title="Sorted by accuracy (low to high)", xlim=(0, 1.05))

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "fig4_knn_accuracy.png"), dpi=150)
plt.show()

# Hardest letters
print("Lowest-accuracy letters:")
for l in sorted_letters[:5]:
    print(f"  '{l}': {per_letter_acc[l]*100:.1f}%")

## 8. Summary figure

In [ ]:
fig = plt.figure(figsize=(16, 10))
gs  = GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)
fig.suptitle("Neural separability analysis — Willett et al. 2021", fontsize=14)

# 1. PCA trajectory (warped, PC1 vs PC2)
ax1 = fig.add_subplot(gs[0, 0])
for i, l in enumerate(SHOW):
    traj = proj_warp[l]
    ax1.plot(traj[:, 0], traj[:, 1], color=colors_show[i],
             label=l, linewidth=2, alpha=0.85)
    ax1.scatter(traj[GO_BIN, 0], traj[GO_BIN, 1],
                color=colors_show[i], s=40, marker="*", zorder=5)
v = pca_warp.explained_variance_ratio_
ax1.set(xlabel=f"PC1 ({v[0]*100:.1f}%)", ylabel=f"PC2 ({v[1]*100:.1f}%)",
        title="PCA trajectory (time-aligned)")
ax1.legend(fontsize=8)

# 2. t-SNE
ax2 = fig.add_subplot(gs[0, 1])
for l in LETTERS:
    mask = y_pool == l
    ax2.scatter(Z[mask, 0], Z[mask, 1],
                color=colors_26[letter_to_i[l]], label=l, s=8, alpha=0.6)
ax2.set(title="t-SNE (all sessions)", xlabel="dim 1", ylabel="dim 2")

# 3. k-NN bars
ax3 = fig.add_subplot(gs[0, 2])
ax3.bar(list(per_letter_acc), [per_letter_acc[l] for l in per_letter_acc],
        color=[cm.RdYlGn(per_letter_acc[l]) for l in per_letter_acc])
ax3.axhline(accuracy, color="red", linestyle="--",
            label=f"{accuracy*100:.1f}%")
ax3.set(title="k-NN accuracy", ylim=(0, 1.05))
ax3.legend(fontsize=9)

# 4. Pen velocity R²
ax4 = fig.add_subplot(gs[1, 0])
ax4.bar(letters_s, [r2_per_letter[l] for l in letters_s],
        color="darkorange", alpha=0.8)
ax4.axhline(mean_r2, color="red", linestyle="--", label=f"mean={mean_r2:.2f}")
ax4.set(title="Pen velocity R² per letter", ylim=(0, 1))
ax4.legend(fontsize=8)

# 5. Decoded velocity example
ax5 = fig.add_subplot(gs[1, 1])
for vt, vp, col, lbl in zip(vel_decoded["a"][0].T, vel_decoded["a"][1].T,
                              ["b", "r"], ["vx", "vy"]):
    ax5.plot(t_ax, vt, f"{col}-",  linewidth=1.4, label=f"true {lbl}")
    ax5.plot(t_ax, vp, f"{col}--", linewidth=1.4, alpha=0.8, label=f"decoded {lbl}")
ax5.axvline(GO_BIN*10, color="k", linestyle=":", alpha=0.5)
ax5.set(xlabel="Time (ms)", title="Decoded pen velocity — 'a'")
ax5.legend(fontsize=7)

# 6. Text summary
ax6 = fig.add_subplot(gs[1, 2])
ax6.axis("off")
summary = (
    f"Data summary\n"
    f"{'-'*28}\n"
    f"Sessions  : {len(ALL_SESSIONS)}\n"
    f"Trials    : {len(y_pool)}\n"
    f"Channels  : {n_chan}\n"
    f"Time bins : {n_time} x 10 ms\n"
    f"Go cue    : bin {GO_BIN} (={GO_BIN*10} ms)\n\n"
    f"Results\n"
    f"{'-'*28}\n"
    f"k-NN LOO  : {accuracy*100:.1f}%\n"
    f"           (paper: 94.1%)\n\n"
    f"Pen vel R²: {total_r2*100:.1f}% of total neural var.\n"
    f"           (paper: ~30%)\n\n"
    f"PCA (warped)\n"
    + "\n".join(f"  PC{i+1}: {v*100:.1f}%"
                for i, v in enumerate(pca_warp.explained_variance_ratio_))
)
ax6.text(0.04, 0.97, summary, transform=ax6.transAxes,
         fontsize=9, verticalalignment="top", fontfamily="monospace",
         bbox=dict(boxstyle="round", facecolor="#fffde7", alpha=0.9))

plt.savefig(os.path.join(OUT_DIR, "fig0_summary.png"), dpi=150, bbox_inches="tight")
plt.show()

print("=" * 50)
print(f"  k-NN LOO accuracy           : {accuracy*100:.1f}%  (paper: 94.1%)")
print(f"  Neural var. explained by vel: {total_r2*100:.1f}%  (paper: ~30%)")
print(f"  PCA explained variance (1-3): " +
      " / ".join(f"{v*100:.1f}%" for v in pca_warp.explained_variance_ratio_))
print("=" * 50)